# Pattern / texture lab

Probe two texture fills and the signals that might tell them apart from text, dashed lines, grids and
ordinary drawing:

- **Stipple**: many small, short vectors spread fairly evenly over an area (dots / specks).
- **Hatching**: many parallel lines packed closely side by side.

The unit is an **element**: one pen-down sub-path of a `Vector`. A CAD hatch or dot fill is often one
`get_drawings()` drawing holding hundreds of disconnected pieces, so every Vector is exploded first
(`_pattern_probe_helpers.explode_elements`). Analysis runs per **(layer, color, width) bucket**
(`SCOPE = 'bucket'`) or per P3 LatestVectorClassification spatial cluster (`SCOPE = 'cluster'`).

**Stipple signals.** Candidates are elements with `size <= DOT_MAX_PT`. Per candidate, over itself + its
`K_NN` nearest candidates:
- `NN-CV`: coefficient of variation of nearest-neighbour distances. Low = even spacing.
- `isotropy`: PCA eigenvalue ratio of the neighbourhood (~1 = 2-D field, ~0 = a dotted / dashed *line*).
- `size CV` and `same shape`: dots in a fill are clones; glyph fragments aren't.

Candidates link into **regions** when closer than `LINK_FACTOR` x the local median NN distance. Each
region gets the **Clark-Evans R** (mean NN distance / its expectation under randomness: > 1 regular,
~1 random, < 1 clumped, like text grouped into words). Verdict = count >= `MIN_DOTS` and
R >= `R_MIN` and isotropy >= `ISO_MIN` and size CV <= `SIZE_CV_MAX`.

**Hatching signals.** Straight elements grouped by angle into families. Collinear pieces merge into one
**track**, so a dashed hatch line counts once. Tracks chain into **runs** when their perpendicular gap is
<= `MAX_SPACING_PT` *and* they overlap along the line by >= `MIN_OVERLAP_FRAC`. That overlap stops an
end-to-end dashed line from looking like hatching. Per run:
- `spacing CV`: regular spacing = hatch; dimension / extension-line bundles are irregular.
- `density ratio`: median spacing / median track length. Hatch << 1; a ladder of ticks or a grid >= 1.
- `dominance`: share of the ink inside the run's bbox along the run's angle (or a cross-hatch partner's).
- `endpoint residual`: how straight the clipped ends are. Informational only, not in the verdict.

Verdict = count >= `MIN_HATCH_LINES` and spacing CV <= `SPACING_CV_MAX` and density <= `DENSITY_MAX`
and dominance >= `DOMINANCE_MIN`.

Every gradient layer runs green -> yellow -> red -> purple over the 1st..99th percentile of its values.
Open the output with `scripts/pipeline_report_viewer.py`.

## 0 - Config

In [ ]:
PDF_PATH = None          # None -> first PDF under references/
PAGE = 0
SCOPE = 'bucket'         # 'bucket' (layer/color/width) or 'cluster' (P3 spatial clusters)
STRAIGHT_TOL_PT = 0.25   # max point distance from the fitted line for an element to count as straight
LINE_WIDTH = 1.0

# Stipple
DOT_MAX_PT = 2.0         # an element whose longer bbox side is <= this is a dot candidate
K_NN = 6                 # neighbourhood size for the per-candidate stats
LINK_FACTOR = 2.5        # region linking distance, x local median NN distance
MIN_DOTS = 20
R_MIN = 1.0              # Clark-Evans R
ISO_MIN = 0.3
SIZE_CV_MAX = 0.3

# Hatching
ANGLE_TOL_DEG = 1.0      # parallel-family angle tolerance (single-linkage, folded to [0, 180))
COLLINEAR_TOL_PT = 0.5   # pieces within this perpendicular offset merge into one track
MAX_SPACING_PT = 6.0     # max gap between neighbouring tracks in a run
MIN_OVERLAP_FRAC = 0.3   # neighbouring tracks must overlap along the line by this share of the shorter one
MIN_HATCH_LINES = 5
SPACING_CV_MAX = 0.25
DENSITY_MAX = 0.5
DOMINANCE_MIN = 0.7
CROSS_MIN_ANGLE_DEG = 10.0

## 1 - Bootstrap

In [ ]:
import colorsys
import math
import os, sys
from pathlib import Path

_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import matplotlib.pyplot as plt
import numpy as np

from rastervec.notebooks import _pattern_probe_helpers as p
from rastervec.notebooks import _vector_probe_helpers as h

if PDF_PATH is None:
    refs = sorted((Path(_root) / 'references').glob('*.pdf'))
    assert refs, 'no PDFs found under references/'
    PDF_PATH = str(refs[0])
print(PDF_PATH, 'page', PAGE)

## 2 - Extract, group, explode into elements

In [ ]:
page_meta, vectors = h.load_page_vectors(PDF_PATH, PAGE)
if SCOPE == 'bucket':
    groups = list(h.bucket_vectors(vectors).items())
elif SCOPE == 'cluster':
    groups = h.cluster_vectors(vectors, page_meta)
else:
    raise ValueError(f'unknown SCOPE {SCOPE!r}')
group_elements = [(key, p.explode_all(members, STRAIGHT_TOL_PT)) for key, members in groups]
all_elements = [e for _, els in group_elements for e in els]
print(f'{len(vectors)} vectors -> {len(groups)} {SCOPE}s -> {len(all_elements)} elements '
      f'({sum(e.line is not None for e in all_elements)} straight)')

report = h.DebugReport('pattern_texture_lab', PDF_PATH, page_meta, line_width=LINE_WIDTH)


def finite(values):
    return [v for v in values if v is not None and math.isfinite(v)]


def gradient_layer(stage, label, units, values):
    # One layer: each unit (a list of elements) coloured by its value, gradient4 over the 1st..99th
    # percentile, drawn low-to-high so the extremes end up on top. nan values are skipped.
    pairs = [(u, v) for u, v in zip(units, values) if v is not None and math.isfinite(v)]
    if not pairs:
        return
    lo, hi = (float(x) for x in np.percentile([v for _, v in pairs], [1, 99]))
    colour, vecs = {}, []
    for u, v in sorted(pairs, key=lambda r: r[1]):
        rgb = h.gradient4(h.normalise(v, lo, hi))
        for e in u:
            colour[id(e.vector)] = rgb
            vecs.append(e.vector)
    report.add_vectors(stage, f'{label} ({lo:.2f}-{hi:.2f})', h.gradient4(1.0), vecs, lambda v: colour[id(v)])


def flat_layer(stage, label, rgb, elements):
    report.add_vectors(stage, label, rgb, [e.vector for e in elements], lambda v: rgb)


def metric_hist(ax, values, title, threshold=None, bins=40):
    values = finite(values)
    if values:
        _, edges, patches = ax.hist(values, bins=bins, edgecolor='white')
        for left, right, patch in zip(edges[:-1], edges[1:], patches):
            patch.set_facecolor(h.gradient4(h.normalise((left + right) / 2, edges[0], edges[-1])))
    if threshold is not None:
        ax.axvline(threshold, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(f'{title} (n={len(values)})')

## 3 - Stipple: candidates, neighbourhood stats, regions

In [ ]:
st_cands, st_stats, st_regions = [], [], []   # st_regions: (group index, StippleRegion, member elements)
for gi, (key, els) in enumerate(group_elements):
    cands = p.stipple_candidates(els, DOT_MAX_PT)
    if not cands:
        continue
    stats = p.neighbourhood_stats(cands, K_NN)
    regions = p.grow_regions(cands, stats, link_factor=LINK_FACTOR, min_dots=MIN_DOTS,
                             r_min=R_MIN, iso_min=ISO_MIN, size_cv_max=SIZE_CV_MAX)
    st_cands.extend(cands)
    st_stats.extend(stats)
    st_regions.extend((gi, r, [cands[i] for i in r.members]) for r in regions)

st_multi = [row for row in st_regions if row[1].count >= 3]
st_hits = [row for row in st_regions if row[1].is_stipple]
print(f'{len(st_cands)} dot candidates -> {len(st_regions)} regions '
      f'({len(st_multi)} with >= 3 dots, {len(st_hits)} stipple verdicts, '
      f'{sum(r.count for _, r, _ in st_hits)} dots)')

flat_layer('stipple', f'candidates (size <= {DOT_MAX_PT:g} pt)', h.GREY, st_cands)
units = [[e] for e in st_cands]
gradient_layer('stipple', 'NN-CV', units, [s.nn_cv for s in st_stats])
gradient_layer('stipple', 'isotropy', units, [s.isotropy for s in st_stats])
gradient_layer('stipple', 'size CV', units, [s.size_cv for s in st_stats])
gradient_layer('stipple', 'same shape frac', units, [s.same_shape_frac for s in st_stats])
gradient_layer('stipple', 'region Clark-Evans R (>= 3 dots)', [m for _, _, m in st_multi],
               [r.clark_evans for _, r, _ in st_multi])
report.add_boxes('stipple', 'region bboxes (>= MIN_DOTS; green = verdict)', '#00b300', [
    (r.bbox, (0.0, 0.7, 0.0) if r.is_stipple else (1.0, 0.55, 0.0))
    for _, r, _ in st_regions if r.count >= MIN_DOTS
])
flat_layer('stipple', 'verdict', h.gradient4(1.0), [e for _, _, m in st_hits for e in m])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
metric_hist(axes[0, 0], [s.nn_cv for s in st_stats], 'candidate NN-CV')
metric_hist(axes[0, 1], [s.isotropy for s in st_stats], 'candidate isotropy', ISO_MIN)
metric_hist(axes[0, 2], [s.size_cv for s in st_stats], 'candidate size CV', SIZE_CV_MAX)
metric_hist(axes[1, 0], [r.clark_evans for _, r, _ in st_multi], 'region Clark-Evans R (>= 3 dots)', R_MIN)
metric_hist(axes[1, 1], [r.isotropy for _, r, _ in st_multi], 'region isotropy (>= 3 dots)', ISO_MIN)
metric_hist(axes[1, 2], [r.size_cv for _, r, _ in st_multi], 'region size CV (>= 3 dots)', SIZE_CV_MAX)
fig.suptitle(f'Stipple -- {Path(PDF_PATH).stem} p{PAGE} (dashed = verdict threshold)')
fig.tight_layout()
report.save_figure(fig, 'histograms_stipple')
plt.show()

print(f"{'grp':>4} {'dots':>5} {'area':>9} {'R':>6} {'iso':>5} {'sizeCV':>6} {'shape':>5}  verdict")
for gi, r, _ in sorted(st_multi, key=lambda row: -row[1].count)[:15]:
    print(f'{gi:>4} {r.count:>5} {r.area:>9.1f} {r.clark_evans:>6.2f} {r.isotropy:>5.2f} '
          f'{r.size_cv:>6.2f} {r.shape_frac:>5.2f}  {"STIPPLE" if r.is_stipple else ""}')

## 4 - Hatching: parallel families, side-by-side runs

In [ ]:
runs_all = []   # (group index, HatchRun)
for gi, (key, els) in enumerate(group_elements):
    runs = p.hatch_runs(els, angle_tol=ANGLE_TOL_DEG, collinear_tol=COLLINEAR_TOL_PT,
                        max_spacing=MAX_SPACING_PT, min_overlap=MIN_OVERLAP_FRAC)
    p.pair_cross_hatch(runs, min_count=MIN_HATCH_LINES, min_angle=CROSS_MIN_ANGLE_DEG)
    p.score_dominance(runs, els, ANGLE_TOL_DEG)
    p.judge_hatch(runs, min_lines=MIN_HATCH_LINES, spacing_cv_max=SPACING_CV_MAX,
                  density_max=DENSITY_MAX, dominance_min=DOMINANCE_MIN)
    runs_all.extend((gi, r) for r in runs)

ht_multi = [(gi, r) for gi, r in runs_all if r.count >= 3]
ht_hits = [(gi, r) for gi, r in runs_all if r.is_hatch]
print(f'{len(runs_all)} runs ({len(ht_multi)} with >= 3 tracks, {len(ht_hits)} hatch verdicts, '
      f'{sum(bool(r.partners) for _, r in ht_hits)} of them cross-hatched)')

units = [r.elements for _, r in ht_multi]
hues = h.golden_hues(len(ht_multi))
fam_colour, fam_vecs = {}, []
for hue, (_, r) in zip(hues, ht_multi):
    rgb = h.hsv_cluster_angle(hue, r.angle)
    for e in r.elements:
        fam_colour[id(e.vector)] = rgb
        fam_vecs.append(e.vector)
report.add_vectors('hatch', 'runs >= 3 tracks (hue=run, value=angle)', h.hsv_cluster_angle(0.6, 90),
                   fam_vecs, lambda v: fam_colour[id(v)])
gradient_layer('hatch', 'track count', units, [r.count for _, r in ht_multi])
gradient_layer('hatch', 'spacing CV', units, [r.spacing_cv for _, r in ht_multi])
gradient_layer('hatch', 'density ratio', units, [r.density_ratio for _, r in ht_multi])
gradient_layer('hatch', 'dominance', units, [r.dominance for _, r in ht_multi])
gradient_layer('hatch', 'endpoint residual (pt)', units, [r.endpoint_residual for _, r in ht_multi])
report.add_boxes('hatch', 'run bboxes (>= MIN_HATCH_LINES; green = verdict)', '#00b300', [
    (r.bbox, (0.0, 0.7, 0.0) if r.is_hatch else (1.0, 0.55, 0.0))
    for _, r in runs_all if r.count >= MIN_HATCH_LINES
])
flat_layer('hatch', 'cross-hatch pairs', (0.0, 0.45, 0.9),
           [e for _, r in runs_all if r.partners for e in r.elements])
flat_layer('hatch', 'verdict', h.gradient4(1.0), [e for _, r in ht_hits for e in r.elements])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
metric_hist(axes[0, 0], [r.count for _, r in ht_multi], 'run track count (>= 3)', MIN_HATCH_LINES)
metric_hist(axes[0, 1], [r.spacing_cv for _, r in ht_multi], 'run spacing CV', SPACING_CV_MAX)
metric_hist(axes[1, 0], [r.density_ratio for _, r in ht_multi], 'run density ratio', DENSITY_MAX)
metric_hist(axes[1, 1], [r.dominance for _, r in ht_multi], 'run dominance', DOMINANCE_MIN)
fig.suptitle(f'Hatching -- {Path(PDF_PATH).stem} p{PAGE} (dashed = verdict threshold)')
fig.tight_layout()
report.save_figure(fig, 'histograms_hatch')
plt.show()

print(f"{'grp':>4} {'n':>4} {'angle':>6} {'med gap':>7} {'gapCV':>6} {'dens':>6} {'dom':>5} {'endRes':>6} {'x':>2}  verdict")
for gi, r in sorted(ht_multi, key=lambda row: -row[1].count)[:15]:
    gap = float(np.median(r.gaps)) if r.gaps else math.nan
    print(f'{gi:>4} {r.count:>4} {r.angle:>6.1f} {gap:>7.2f} {r.spacing_cv:>6.2f} {r.density_ratio:>6.2f} '
          f'{r.dominance:>5.2f} {r.endpoint_residual:>6.2f} {len(r.partners):>2}  {"HATCH" if r.is_hatch else ""}')

## 5 - Context + write manifest

In [ ]:
claimed = {id(e) for _, _, m in st_hits for e in m} | {id(e) for _, r in ht_hits for e in r.elements}
flat_layer('context', 'everything else', h.LIGHT_GREY, [e for e in all_elements if id(e) not in claimed])

out = report.finish()
print('wrote', out)
for e in report.layers:
    print(f"  {e['stage']:>10} | {e['layer']}")
print()
print('open with:')
print(report.viewer_command())